# The Data Ingestion Pipeline. 
- This stage is where raw information (PDFs, text files, etc.) is transformed into a searchable mathematical format.

## 1. Data Ingestion & Parsing: 
- Extracting raw text from various file formats while maintaining document structure.

In [9]:
###Document Structure
# !pip3 install -U langchain langchain-core langchain-community
import warnings
warnings.filterwarnings("ignore")

from langchain_core.documents import Document

In [10]:
doc=Document(
    page_content="this is the main text content",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Nazmul Alam",
        "date_created":"2026-05-14"
    }
)


In [11]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [12]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [13]:
### TextLoader
# from langchain.document_loaders import TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()


In [14]:
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [15]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [16]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
# print(pdf_documents)

## 2. Chunking Strategy: 
- Breaking down massive documents into smaller, semantically meaningful "chunks" to fit within the context windows of Embedding and LLM models.

In [17]:
# !pip3 install langchain-text-splitters
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: PublicWaterMassMailing.pdf
  ✓ Loaded 8 pages

Processing: Non-text-searchable.pdf
  ✓ Loaded 1 pages

Processing: 1706.03762v7.pdf
  ✓ Loaded 15 pages

Total documents loaded: 24


In [18]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks=split_documents(all_pdf_documents)
# print(chunks)

Split 24 documents into 52 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': '1706.03762v7.pdf', 'file_type': 'pdf'}


## 3. Embeddings: 
- Utilizing models like `all-MiniLM-L6-v2` via Hugging Face to convert text into 384-dimensional vectors

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
print(embedding_manager)

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


## 4. Vector Database (ChromaDB): 
- Storing these vectors and their associated metadata (source, page number, author, etc) in a persistent database to enable efficient similarity searches.

In [22]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()

  # print(vectorstore)  

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 52


In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 52 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings with shape: (52, 384)
Adding 52 documents to vector store...
Successfully added 52 documents to vector store
Total documents in collection: 104


## 5. Retriever Pipeline From VectorStore
- Implementing a custom `RAGRetriever` class that takes a user query, embeds it, and performs a similarity search to return the most relevant context blocks.

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_84dc862b_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'producer': 'pdfTeX-1.40.25',
   'source_file': '1706.03762v7.pdf',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'moddate': '2024-04-10T21:11:43+00:00',
   'trapped': '/False',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'content_length': 216,
   'subject': '',
   'file_type': 'pdf',
   'total_pages': 15,
   'source': '../data/pdf/1706.03762v7.pdf',
   'page': 2,
   'page_label': '3',
   'creator': 'LaTeX with hyperref',
   'doc_index': 12,
   'author': '',
   'title': '',
   'keywords': ''},
  'similarity_score': 0.13995492458343506,
  'distance': 0.8600450754165649,
  'rank': 1},
 {'id': 'doc_14b31361_12',
  'content': '3.2 Attention\n

In [27]:
rag_retriever.retrieve('What is Recurrent neural networks')

Retrieving documents for query: 'What is Recurrent neural networks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_561c9597_4',
  'content': '1 Introduction\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks\nin particular, have been firmly established as state of the art approaches in sequence modeling and\ntransduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous\nefforts have since continued to push the boundaries of recurrent language models and encoder-decoder\narchitectures [38, 24, 15].\nRecurrent models typically factor computation along the symbol positions of the input and output\nsequences. Aligning the positions to steps in computation time, they generate a sequence of hidden\nstates ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently\nsequential nature precludes parallelization within training examples, which becomes critical at longer\nsequence lengths, as memory constraints limit batching across examples. Recent work has achieved',
  'metadata': 

## 6. VectorDB To LLM Output Generation



In [28]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("API_KEY"))

True

In [29]:
# !pip3 install langchain-core langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [30]:


class OpenAILLM:
    def __init__(self, model_name: str = "gpt-4o", api_key: str = None):
        """
        Initialize OpenAI LLM
        
        Args:
            model_name: OpenAI model name (gpt-4o, gpt-4-turbo, gpt-3.5-turbo)
            api_key: OpenAI API key (or set OPENAI_API_KEY environment variable)
        """
        self.model_name = model_name
        # Standardize on OPENAI_API_KEY environment variable
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY")
        
        if not self.api_key:
            raise ValueError(
                "OpenAI API key is required. Set OPENAI_API_KEY environment variable "
                "or pass api_key parameter."
            )
        
        # Initialize the LangChain OpenAI wrapper
        self.llm = ChatOpenAI(
            api_key=self.api_key,
            model=self.model_name,
            temperature=0.1,  # Critical for RAG to maintain factual grounding
            max_tokens=1024
        )
        
        print(f"Initialized OpenAI LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context via a structured prompt.
        """
        # Define an expert-level RAG prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are an expert AI assistant specializing in Software Engineering and Machine Learning. 
Use the following retrieved context to answer the user's question with technical depth and clarity.

Context:
{context}

Question: {question}

Answer: 
Provide a precise answer based ONLY on the context provided. If the information is not present, 
state that the context does not provide sufficient data."""
        )
        
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            # Professional error logging would typically go here
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Lightweight response generation for low-latency tasks.
        """
        simple_prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [31]:
import os

# Initialize OpenAI LLM (Ensure OPENAI_API_KEY is set in your environment)
try:
    # Initialize using the refactored OpenAILLM class
    # Defaulting to 'gpt-4o' for high-reasoning tasks
    openai_llm = OpenAILLM(
        model_name="gpt-4o", 
        api_key=os.getenv("API_KEY")
    )
    print("OpenAI LLM initialized successfully!")

except ValueError as e:
    print(f"Configuration Error: {e}")
    print("Action Required: Please set your OPENAI_API_KEY environment variable.")
    openai_llm = None

except Exception as e:
    print(f"An unexpected error occurred during initialization: {e}")
    openai_llm = None

Initialized OpenAI LLM with model: gpt-4o
OpenAI LLM initialized successfully!


In [33]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve('What is Recurrent neural networks')

Retrieving documents for query: 'What is Recurrent neural networks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_561c9597_4',
  'content': '1 Introduction\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks\nin particular, have been firmly established as state of the art approaches in sequence modeling and\ntransduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous\nefforts have since continued to push the boundaries of recurrent language models and encoder-decoder\narchitectures [38, 24, 15].\nRecurrent models typically factor computation along the symbol positions of the input and output\nsequences. Aligning the positions to steps in computation time, they generate a sequence of hidden\nstates ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently\nsequential nature precludes parallelization within training examples, which becomes critical at longer\nsequence lengths, as memory constraints limit batching across examples. Recent work has achieved',
  'metadata': 

## 7. Integration Vectordb Context pipeline With LLM output

In [34]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()

### 1. Initialize the OpenAI LLM
# Using OpenAI API Key from environment
openai_api_key = os.getenv("API_KEY")

if not openai_api_key:
    raise ValueError("API_KEY not found in environment variables.")

llm = ChatOpenAI(
    api_key=openai_api_key,
    model="gpt-4o",  # or "gpt-3.5-turbo"
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    """
    Standard RAG pipeline:
    1. Retrieve: Fetches relevant documents.
    2. Augment: Constructs a prompt with the context.
    3. Generate: Invokes the LLM for the answer.
    """
    
    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    
    # Process results: handling both dictionary and Document object formats
    context_list = []
    for doc in results:
        # Check if doc is a dictionary or a LangChain Document object
        content = doc.get('content') if isinstance(doc, dict) else getattr(doc, 'page_content', "")
        if content:
            context_list.append(content)
            
    context = "\n\n".join(context_list)
    
    if not context:
        return "No relevant context found to answer the question."
    
    # Generate the answer using OpenAI LLM
    # Note: We use f-strings for the prompt and wrap it in a HumanMessage
    prompt = f"""Use the following context to answer the question concisely.
    
Context:
{context}

Question: {query}

Answer:"""
    
    try:
        # In newer LangChain versions, we pass a list of messages to invoke
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        return f"An error occurred during generation: {str(e)}"

In [35]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
An attention mechanism is a function that maps a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, with the weights determined by the compatibility of the query with the corresponding key. This mechanism allows the model to focus on different parts of the input sequence, capturing long-distance dependencies effectively.


## 8. Enhanced RAG Pipeline Features

In [36]:
import os
from langchain_core.messages import HumanMessage

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    Enhanced RAG pipeline for OpenAI:
    - Returns answer, structured sources, confidence score, and optional full context.
    - Implements score thresholding for ML rigor.
    """
    
    # 1. Retrieve with thresholding
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {
            'answer': 'No relevant context found within the specified similarity threshold.', 
            'sources': [], 
            'confidence': 0.0, 
            'context': ''
        }
    
    # 2. Process Context and Metadata
    context_parts = []
    sources = []
    
    for doc in results:
        # Standardize content and metadata access
        content = doc.get('content', '')
        metadata = doc.get('metadata', {})
        score = doc.get('similarity_score', 0.0)
        
        context_parts.append(content)
        
        # Build structured source attribution
        sources.append({
            'source': metadata.get('source_file', metadata.get('source', 'unknown')),
            'page': metadata.get('page', 'unknown'),
            'score': round(score, 4),
            'preview': content[:300] + '...' if content else ""
        })
    
    context = "\n\n".join(context_parts)
    confidence = max([s['score'] for s in sources]) if sources else 0.0
    
    # 3. Generate Answer using OpenAI LLM
    # Defining a more technical prompt persona
    prompt_text = f"""You are an expert assistant. Use the following context to answer the question concisely and accurately.
    
Context:
{context}

Question: {query}

Answer:"""

    try:
        # Utilizing the updated message-based invocation
        response = llm.invoke([HumanMessage(content=prompt_text)])
        
        output = {
            'answer': response.content,
            'sources': sources,
            'confidence': confidence
        }
        
        if return_context:
            output['context'] = context
            
        return output

    except Exception as e:
        return {
            'answer': f"Error during generation: {str(e)}",
            'sources': sources,
            'confidence': confidence,
            'context': context if return_context else ""
        }

# --- Example usage with OpenAI ---
result = rag_advanced("Hard Negative Mining Techniques", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print(result)

Retrieving documents for query: 'Hard Negative Mining Techniques'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
{'answer': 'No relevant context found within the specified similarity threshold.', 'sources': [], 'confidence': 0.0, 'context': ''}


In [37]:
import time
from typing import List, Dict, Any
from langchain_core.messages import HumanMessage, SystemMessage

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        """
        Initialize the Advanced RAG Pipeline.
        Args:
            retriever: The vector store retrieval component.
            llm: An instance of ChatOpenAI.
        """
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Maintain conversation state

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, 
              stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        
        # 1. Context Retrieval
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        if not results:
            answer = "I'm sorry, but I couldn't find any relevant information in the documentation to answer that."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc.get('content', '') for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': round(doc.get('similarity_score', 0.0), 4),
                'preview': doc.get('content', '')[:120] + '...'
            } for doc in results]

            # 2. Augmented Prompt Construction
            system_instruction = "You are a technical expert. Answer the question precisely using the provided context."
            user_prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
            
            messages = [
                SystemMessage(content=system_instruction),
                HumanMessage(content=user_prompt)
            ]

            # 3. Generation (Streaming vs. Blocking)
            if stream:
                print("Streaming response: ", end="", flush=True)
                full_response = ""
                # OpenAI's .stream() returns a generator of tokens
                for chunk in self.llm.stream(messages):
                    content = chunk.content
                    print(content, end="", flush=True)
                    full_response += content
                print("\n")
                answer = full_response
            else:
                response = self.llm.invoke(messages)
                answer = response.content

        # 4. Citation Logic
        citations = [f"[{i+1}] {src['source']} (Page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = f"{answer}\n\nSources:\n" + "\n".join(citations) if citations else answer

        # 5. Summarization (Post-Processing)
        summary = None
        if summarize and answer and results:
            summary_prompt = f"Provide a technical 2-sentence summary of this explanation:\n{answer}"
            summary_resp = self.llm.invoke([HumanMessage(content=summary_prompt)])
            summary = summary_resp.content

        # 6. Session Persistence
        current_turn = {
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary,
            'timestamp': time.time()
        }
        self.history.append(current_turn)

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# --- Example Usage ---
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is the Attention mechanism?", stream=True, summarize=True)

Retrieving documents for query: 'What is the Attention mechanism?'
Top K: 5, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming response: The attention mechanism is a process used in machine learning models, particularly in natural language processing, to focus on specific parts of the input data when generating an output. It involves mapping a query and a set of key-value pairs to an output, where all these elements are vectors. The output is calculated as a weighted sum of the values, with the weights determined by the compatibility of the query with the corresponding keys. This allows the model to dynamically prioritize different parts of the input data, enhancing its ability to capture relevant information and improve performance on tasks such as translation, summarization, and more.

